In [1]:
from datasets import load_dataset, DatasetDict
dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import BertTokenizer, BertModel

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# Import BERT pre-trained model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased').to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [7]:
text = 'Replace me with any text you like.'
tokens = tokenizer(text, return_tensors='pt').to(device)
tokens

output = bert(**tokens, output_hidden_states=True)
output.hidden_states[0].detach().shape

torch.Size([1, 10, 768])

In [8]:
output = bert(**tokens, output_hidden_states=True, output_attentions=True, return_dict=True)
output.hidden_states[0]

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


tensor([[[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
         [-0.2368,  0.3720,  0.9008,  ...,  0.0719,  1.2566,  0.4192],
         [ 0.0120,  0.3938, -0.8486,  ...,  0.6180,  0.9804,  0.1418],
         ...,
         [ 0.3810,  0.4720,  0.4045,  ...,  0.1828,  1.0139,  0.2938],
         [-0.1550,  0.0692, -0.1660,  ...,  0.4387,  0.6441,  0.5938],
         [-0.1474, -0.0411, -0.0732,  ..., -0.1157,  0.0421, -0.0550]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>)

# Create an LLM model using BERT

In [9]:
# Create a new LLM model with a new head
class BertForBinaryClassification(nn.Module):
  def __init__(self, num_labels=2):
    super(BertForBinaryClassification, self).__init__()

    # Load the pre-trained BERT model
    self.bert = BertModel.from_pretrained('bert-base-uncased')

    # Add a new classification head
    self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    self.dropout = nn.Dropout(self.bert.embeddings.dropout.p)

    # Initialize the weights and biases
    nn.init.xavier_normal_(self.classifier.weight)
    nn.init.zeros_(self.classifier.bias)

  def forward(self, input_ids, attention_mask=None, token_type_ids=None):
    # Pass the input through the BERT model
    outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

    # Extract the pooled output and apply dropout
    pooled_output = outputs.pooler_output
    pooled_output = self.dropout(pooled_output)

    # Pass the pooled output through the classification head
    logits = self.classifier(pooled_output)

    return logits



In [10]:
# Create an instance of the binary classification model
model = BertForBinaryClassification(num_labels=2).to(device)

# test the output
tokens = tokenizer(text, return_tensors='pt')
tokens = tokens.to(device)
output = model(**tokens)
output

tensor([[-0.8422,  1.1444]], device='cuda:0', grad_fn=<AddmmBackward0>)

### Explaining the Model's Output

The output from the `BertForBinaryClassification` model provides raw scores (logits) for each class, which are then converted into probabilities.

*   **Logits**: These are the direct numerical outputs from the final layer of the neural network before any activation function like softmax is applied. In our case, `tensor([[2.7416, 0.4256]])` means the model assigned a score of `2.7416` to the first class and `0.4256` to the second class for the given input text.

*   **Probabilities**: After applying the `softmax` function to the logits, we get `tensor([[0.9102, 0.0898]])`. These values represent the predicted probabilities for each class. They sum up to 1 (or very close to it due to floating-point precision).
    *   `0.9102` (or 91.02%) is the model's confidence that the input belongs to the first class.
    *   `0.0898` (or 8.98%) is the model's confidence that the input belongs to the second class.

In a binary classification task, if the first class typically represents a positive sentiment (e.g., 'positive review') and the second class a negative sentiment (e.g., 'negative review'), then for the input `'Replace me with any text you like.'`, the model strongly predicts it belongs to the positive class.

In [11]:
probabilities = torch.softmax(output, dim=1)
print('Logits:', output)
print('Probabilities:', probabilities)

Logits: tensor([[-0.8422,  1.1444]], device='cuda:0', grad_fn=<AddmmBackward0>)
Probabilities: tensor([[0.1206, 0.8794]], device='cuda:0', grad_fn=<SoftmaxBackward0>)


# Import the dataset from IMDB

In [12]:
# Load IMDB dataset from HF
dataset = load_dataset('imdb')
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [13]:
# Tokenization function to process each data sample
def tokenize_function(examples):
  return tokenizer(examples['text'],
                   max_length=512,
                   padding='max_length',
                   truncation=True)


# Create a small dataset with a mixture of postive and negative reviews
sample_dataset = DatasetDict({split: dataset[split].select(range(10000, 15000)) for split in ['train','test']})

# Apply tojenize_function to the dataset
tokenized_datasets = sample_dataset.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
})

In [14]:
# Remove the text columns
tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
})

In [15]:
# Change the format to PyToch Tensors
tokenized_datasets.set_format('torch') #, columns=['input_ids','attention_mask','label'])
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
})

In [16]:
# Create DataLoaders
train_dataloader = DataLoader(tokenized_datasets['train'], shuffle=True, batch_size=32)
test_dataloader = DataLoader(tokenized_datasets['test'], shuffle=False, batch_size=32)

In [17]:
X = next(iter(train_dataloader))
X['input_ids'].shape

torch.Size([32, 512])

# Finetune the model

In [18]:
# Loss and Optimizer functions
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=.01)

In [19]:
# get a batch of data and pass it through the model
batch = next(iter(train_dataloader))
batch = {k: v.to(device) for k, v in batch.items()}

# clear the gradients
optimizer.zero_grad(set_to_none=True)

# Prepare inputs for the model (excluding 'label')
model_inputs = {
    'input_ids': batch['input_ids'],
    'attention_mask': batch['attention_mask'],
    'token_type_ids': batch['token_type_ids']
}

# forward pass
output = model(**model_inputs)
predicted_labels = torch.argmax(output, dim=1)

loss = loss_fn(output, batch['label'])
accuracy = (predicted_labels == batch['label']).float().mean()

# backward pass
loss.backward()
optimizer.step()

In [20]:
print('Model predictions:',predicted_labels)
print('True labels', batch['label'])
print(f'Loss: {loss.item():.2f}')
print(f'Accuracy: {100*accuracy.item():.1f} %')

Model predictions: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')
True labels tensor([0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0,
        0, 1, 0, 0, 0, 1, 0, 0], device='cuda:0')
Loss: 0.93
Accuracy: 34.4 %


# Train the model

In [21]:
# To train the model select 15,000 reviews - 7,500 for each dataset
sample_dataset = DatasetDict({split: dataset[split].select(range(5000, 20000)) for split in ['train','test']})

In [22]:
# Apply tojenize_function to the dataset
tokenized_datasets = sample_dataset.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15000
    })
})

In [23]:
# Remove the text columns
tokenized_datasets = tokenized_datasets.remove_columns(['text'])

# Change the format to PyToch Tensors
tokenized_datasets.set_format('torch')

# Create DataLoaders
train_dataloader = DataLoader(tokenized_datasets['train'], shuffle=True, batch_size=32)
test_dataloader = DataLoader(tokenized_datasets['test'], shuffle=False, batch_size=32)


In [24]:
# Create an instance of the binary classification model
model = BertForBinaryClassification(num_labels=2).to(device)

In [25]:
# Count the total number of parameters in the model
print(f'Total trainable paremeters in bert: {sum([torch.numel(params) for params in model.parameters() if params.requires_grad]):,}')

Total trainable paremeters in bert: 109,483,778


In [26]:
# Freeze the embeddings and attention layers
for name, param in model.named_parameters():
  if any(substring in name for substring in ['embeddings', 'attention']):
    param.requires_grad = False


# Count the number of trainable and non-trainable parameters
total_params = sum(p.numel() for p in model.parameters())
total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_non_trainable_params = total_params - total_trainable_params

print(f'There are {total_non_trainable_params:,} ({100* total_non_trainable_params/total_params:.1f}%) frozen weights,')
print(f'\tand {total_trainable_params:,} ({100*total_trainable_params/total_params:.1f}%) trainable weights.')

There are 52,204,032 (47.7%) frozen weights,
	and 57,279,746 (52.3%) trainable weights.


In [27]:
# Loss and Optimizer functions
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=.01)

In [28]:
# get a batch of data and pass it through the model
numsteps = 300

train_loss = []
test_loss = []

train_accuracy = []
test_accuracy = []

for step in range(numsteps):

  step_loss = 0
  step_accuracy = 0

  model.train()

  for batch in train_dataloader:

    batch = {k: v.to(device) for k, v in batch.items()}

    # clear the gradients
    optimizer.zero_grad(set_to_none=True)

    # Prepare inputs for the model (excluding 'label')
    model_inputs = {
        'input_ids': batch['input_ids'],
        'attention_mask': batch['attention_mask'],
        'token_type_ids': batch['token_type_ids']
    }

    # forward pass
    output = model(**model_inputs)
    predicted_labels = torch.argmax(output, dim=1)

    loss = loss_fn(output, batch['label'])
    accuracy = (predicted_labels == batch['label']).float().mean()

    step_loss += loss.item()
    step_accuracy += accuracy.item()

    # backward pass
    loss.backward()
    optimizer.step()

  train_loss.append(step_loss/len(train_dataloader))
  train_accuracy.append(step_accuracy/len(train_dataloader))

  step_loss = 0
  step_accuracy = 0

  if step % 10 == 0:

    model.eval()
    with torch.no_grad():
      for batch in test_dataloader:

        batch = {k: v.to(device) for k, v in batch.items()}
        model_inputs = {
            'input_ids': batch['input_ids'],
            'attention_mask': batch['attention_mask'],
            'token_type_ids': batch['token_type_ids']
        }
        output = model(**model_inputs)
        predicted_labels = torch.argmax(output, dim=1)

        loss = loss_fn(output, batch['label'])
        accuracy = (predicted_labels == batch['label']).float().mean()
        step_loss += loss.item()
        step_accuracy += accuracy.item()

    test_loss.append(step_loss/len(test_dataloader))
    test_accuracy.append(step_accuracy/len(test_dataloader))

    print(f'Step {step+1}/{numsteps} | losses (train/test): {train_loss[-1]:.2f}/{test_loss[-1]:.2f} | accuracy (train/test): {100*train_accuracy[-1]:.2f}%/{100*test_accuracy[-1]:.2f}%')



Step 1/300 | losses (train/test): 0.32/0.20 | accuracy (train/test): 85.68%/92.29%
Step 11/300 | losses (train/test): 0.01/0.35 | accuracy (train/test): 99.66%/93.01%
Step 21/300 | losses (train/test): 0.01/0.37 | accuracy (train/test): 99.77%/93.21%


KeyboardInterrupt: 